# 07. 동적 페이지와 Selenium — 2019년 코드가 지금 안 되는 이유

`legacy/News screpion.ipynb` 의 첫 줄은 이랬다.

```python
driver = webdriver.Chrome('C:/App/chromedriver_win32/chromedriver')
```

이 한 줄이 **Selenium 4 에서는 예외**다. 그리고 애초에 Selenium 이 필요한 작업도 아니었다.

## 학습 목표
1. 브라우저 자동화가 **정말 필요한지** 판별하는 법
2. Selenium 2/3 문법 → 4.x 대조표 (무엇이 삭제됐는가)
3. 명시적 대기(`WebDriverWait`) — `time.sleep` 을 쓰지 않는 이유
4. 리소스 정리(`quit`)와 headless 옵션
5. 더 나은 대안: 내부 API(XHR) 직접 호출

> 이 노트북은 **브라우저를 실제로 띄우지 않아도** 끝까지 읽히도록 썼다.
> Selenium 이 설치돼 있으면 문법 검증까지 하고, 없으면 해당 셀만 건너뛴다.

## 1. 먼저 물어야 할 것 — 정말 브라우저가 필요한가

브라우저 자동화는 **가장 느리고 가장 잘 깨지는** 수집 방법이다. 순서대로 확인한다.

| 순서 | 확인 | 가능하면 |
|:--:|---|---|
| 1 | 공식 **API** 가 있는가 | API 사용 (가장 안정적·합법적) |
| 2 | `requests` 응답 HTML 에 원하는 데이터가 있는가 | requests + BeautifulSoup (06 노트북) |
| 3 | 개발자도구 Network 탭에 **JSON(XHR)** 요청이 보이는가 | 그 URL 을 직접 호출 |
| 4 | 로그인·무한스크롤·JS 렌더링이 필수인가 | 그제서야 Selenium/Playwright |

2번 판별은 간단하다. 브라우저에서 보이는 문자열이 `requests` 응답에도 있으면 정적 페이지다.

In [1]:
def looks_static(html: str, needle: str) -> bool:
    """브라우저에 보이는 문자열이 원본 HTML 에도 있으면 정적 페이지다."""
    return needle in html


sample_static = "<html><body><li class='news-item'>RPA 도입 기업 늘어</li></body></html>"
sample_dynamic = "<html><body><div id='root'></div><script src='/app.js'></script></body></html>"

print("정적 페이지 판정:", looks_static(sample_static, "RPA"))
print("동적 페이지 판정:", looks_static(sample_dynamic, "RPA"), " ← 본문이 비어 있고 script 만 있다")

정적 페이지 판정: True
동적 페이지 판정: False  ← 본문이 비어 있고 script 만 있다


원본이 수집하려던 2019년 포털 뉴스 검색 결과는 **서버가 HTML 을 완성해 보내는 정적 페이지**였다.
즉 Selenium 없이 `requests` 로 충분했고, 브라우저를 띄운 만큼 느리고 불안정해졌다.

## 2. Selenium 2/3 → 4 대조표

| 2019년 코드 | 지금 (Selenium 4.6+) | 비고 |
|---|---|---|
| `webdriver.Chrome('/path/chromedriver')` | `webdriver.Chrome()` | 경로 인자 **삭제**. Selenium Manager 가 드라이버를 자동 관리 |
| `webdriver.Chrome(executable_path=...)` | `webdriver.Chrome(service=Service(path))` | 굳이 지정하려면 `Service` 객체로 |
| `driver.find_element_by_id("x")` | `driver.find_element(By.ID, "x")` | `find_element_by_*` 전부 **삭제** |
| `driver.find_elements_by_class_name(...)` | `driver.find_elements(By.CLASS_NAME, ...)` | 복수형도 동일 |
| `time.sleep(3)` | `WebDriverWait(driver, 10).until(...)` | 고정 대기 → 조건 대기 |
| `options.headless = True` | `options.add_argument("--headless=new")` | 속성 방식 제거 |
| `driver.close()` | `driver.quit()` | `close` 는 창 하나만, `quit` 이 프로세스 종료 |

특히 첫 줄이 중요하다. **Selenium 4.6 부터 chromedriver 를 직접 내려받을 필요가 없다.**
2019년에 가장 귀찮던 "크롬 버전과 드라이버 버전 맞추기" 문제가 사라졌다.

In [2]:
try:
    import selenium
    from selenium import webdriver  # noqa: F401

    SELENIUM = True
    print("selenium", selenium.__version__, "— 설치됨")
except ImportError:
    SELENIUM = False
    print("selenium 미설치 — 문법 설명만 진행한다 (pip install selenium)")

selenium 미설치 — 문법 설명만 진행한다 (pip install selenium)


## 3. 지금 문법으로 다시 쓴 수집기

아래는 원본 `DefaultCrw()` 를 Selenium 4 문법 + 명시적 대기 + 자원 정리로 다시 쓴 것이다.
**실행은 하지 않는다**(브라우저·네트워크·대상 사이트 약관이 필요하다). 구조를 읽는 것이 목적이다.

In [3]:
SCRAPER_SOURCE = '''
from contextlib import contextmanager

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait


@contextmanager
def chrome(headless: bool = True):
    """with 블록을 벗어나면 반드시 quit — 좀비 크롬 프로세스를 막는다."""
    options = Options()
    if headless:
        options.add_argument("--headless=new")     # 구식: options.headless = True
    options.add_argument("--window-size=1280,900")
    options.add_argument("--lang=ko-KR")
    driver = webdriver.Chrome(options=options)      # 드라이버 경로 인자 없음 (Selenium Manager)
    try:
        yield driver
    finally:
        driver.quit()


def collect(keyword: str, item_selector: str, link_selector: str, timeout: float = 10.0) -> list[dict]:
    """검색 결과 목록에서 제목·링크를 수집한다.

    선택자를 인자로 받는 이유: 사이트 DOM 은 반드시 바뀐다.
    바뀔 값을 코드 안쪽에 박아 두면 그때마다 함수를 고쳐야 한다.
    """
    url = f"https://example.test/search?q={keyword}"
    with chrome() as driver:
        driver.get(url)

        # 고정 sleep 대신 '조건이 만족될 때까지' 기다린다.
        # 빠른 날에는 즉시 진행하고, 느린 날에도 timeout 까지는 버틴다.
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, item_selector))
        )

        rows = []
        for node in driver.find_elements(By.CSS_SELECTOR, item_selector):
            try:
                link = node.find_element(By.CSS_SELECTOR, link_selector)
            except Exception:            # 항목마다 구조가 다를 수 있다 — 하나 때문에 멈추지 않는다
                continue
            rows.append({"title": link.text.strip(), "url": link.get_attribute("href")})
        return rows
'''
print(SCRAPER_SOURCE)


from contextlib import contextmanager

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait


@contextmanager
def chrome(headless: bool = True):
    """with 블록을 벗어나면 반드시 quit — 좀비 크롬 프로세스를 막는다."""
    options = Options()
    if headless:
        options.add_argument("--headless=new")     # 구식: options.headless = True
    options.add_argument("--window-size=1280,900")
    options.add_argument("--lang=ko-KR")
    driver = webdriver.Chrome(options=options)      # 드라이버 경로 인자 없음 (Selenium Manager)
    try:
        yield driver
    finally:
        driver.quit()


def collect(keyword: str, item_selector: str, link_selector: str, timeout: float = 10.0) -> list[dict]:
    """검색 결과 목록에서 제목·링크를 수집한다.

    선택자를 인자로 받는 이유: 사이트 DOM 은 반드시 바뀐다.
    바뀔 값을 코드 안쪽에 박아 두면 그때마다 함수를 고쳐야 한다.
    """
    url = f"ht

In [4]:
# 문법이 유효한지만 확인한다 (실행하지 않는다)
import ast

ast.parse(SCRAPER_SOURCE)
print("구문 검사 통과 — 위 코드는 그대로 .py 로 옮겨 쓸 수 있다")

구문 검사 통과 — 위 코드는 그대로 .py 로 옮겨 쓸 수 있다


### 원본 대비 무엇이 달라졌나

| 원본 | 개선 |
|---|---|
| 드라이버 경로 하드코딩(`C:/App/...`) | Selenium Manager 자동 관리 |
| 전역 `driver` — 노트북이 끝나도 살아 있음 | `contextmanager` 로 반드시 `quit()` |
| 대기 없음(간헐적 0건) | `WebDriverWait` 조건 대기 |
| 선택자 함수 안에 하드코딩 | 인자로 주입 |
| 항목 하나 실패 시 전체 중단 | 항목 단위 예외 처리 |
| 함수가 전역 `filePaht` 를 읽음(오타 변수) | 지역 변수·명시적 반환 |

## 4. 명시적 대기 — `sleep` 을 쓰지 않는 이유

```python
driver.get(url); time.sleep(3)       # 3초면 되겠지
```
네트워크가 느린 날엔 3초로 부족해 **간헐적으로 0건**이 되고, 빠른 날엔 3초를 낭비한다.
조건 대기는 "요소가 나타나면 즉시" 진행한다.

| 조건 | 언제 |
|---|---|
| `presence_of_element_located` | DOM 에 존재 (보이지 않아도 됨) |
| `visibility_of_element_located` | 화면에 보임 |
| `element_to_be_clickable` | 클릭 가능 |
| `text_to_be_present_in_element` | 특정 텍스트가 채워짐 |
| `staleness_of(old)` | 페이지가 실제로 갈아끼워짐 (페이지 전환 확인의 정석) |

## 5. 더 나은 대안 — 내부 API 직접 호출

동적 페이지라도 데이터는 결국 **어딘가의 JSON** 에서 온다.
개발자도구 → Network → Fetch/XHR 에서 그 요청을 찾으면 브라우저 없이 끝난다.

```python
response = requests.get(
    "https://example.test/api/search",
    params={"q": "업무 자동화", "page": 1},
    headers={"User-Agent": "...", "Referer": "https://example.test/"},
    timeout=5,
)
data = response.json()["items"]        # 파싱 불필요, 훨씬 빠르고 안 깨진다
```

장점: 수십 배 빠르고, DOM 변경에 영향받지 않으며, 페이지네이션이 파라미터로 명확하다.
주의: 비공개 내부 API 는 예고 없이 바뀌고, **이용약관상 허용 여부는 별개**다 (06 노트북 §8).

## 6. Playwright — 요즘의 기본 선택

새로 시작한다면 Selenium 대신 Playwright 를 먼저 검토할 만하다.

| | Selenium | Playwright |
|---|---|---|
| 드라이버 | Selenium Manager 가 관리 | `playwright install` 한 번 |
| 대기 | 직접 `WebDriverWait` | **자동 대기** 내장 |
| 비동기 | 별도 | async 기본 지원 |
| 네트워크 가로채기 | 제한적 | 기본 기능 (XHR 응답 직접 수집 가능) |

다만 Selenium 은 자료가 압도적으로 많고 레거시 코드가 대부분 Selenium 이라, **읽을 줄은 알아야 한다.**

## 정리

* Selenium 은 **마지막 수단**이다. API → 정적 HTML → 내부 XHR → 브라우저 순으로 검토한다
* `find_element_by_*`, 드라이버 경로 인자, `options.headless` 는 **삭제된 문법**이다
* `time.sleep` 대신 `WebDriverWait`, `close` 대신 `quit`
* 선택자는 코드 바깥으로 (반드시 바뀐다)

다음: **08. 주피터에서 OpenCV** — 이 저장소 노트북 대부분이 밟은 `cv2.imshow` 문제.